Notebook realizado por Juan David Sánchez. Contacto: juansanchez@unam.edu

# **Tarea 1: Manejo de DataFrame y cálculo de pIC50**
__Target: Acetylcholinesterase (CHEMBL220)__

Versión de Python: 3.14.3 (VS Code)

Nota: El código se realizará en inglés (a exepción de sus anotaciones y prints) con el fin de reutilizarlo en un futuro.

In [67]:
# Importing necessary libraries

import pandas as pd
import numpy as np
import math

**Paso 1: Cargar el dataset (archivo .tsv) y análisis exploratio inicial de los datos**

Nota: Debido a que el archivo es .tsv tenemos que tener en cuenta el sep='\t'

In [68]:
# Loading the dataset
df = pd.read_csv('ache_chembl220_raw.tsv', sep='\t')

# Saving the initial size of the dataset
size_step1 = df.shape

# Printing the initial size and columns of the dataset
print(f'🔶 El dataset inicialmente tiene {size_step1[0]} filas y {size_step1[1]} columnas')
print(f'🔷 Las columnas del dataset son: {list(df.columns)}')

df.sample(5)


🔶 El dataset inicialmente tiene 20132 filas y 48 columnas
🔷 Las columnas del dataset son: ['Molecule ChEMBL ID', 'Molecule Name', 'Molecule Max Phase', 'Molecular Weight', '#RO5 Violations', 'AlogP', 'Compound Key', 'Smiles', 'Standard Type', 'Standard Relation', 'Standard Value', 'Standard Units', 'pChEMBL Value', 'Data Validity Comment', 'Comment', 'Uo Units', 'Ligand Efficiency BEI', 'Ligand Efficiency LE', 'Ligand Efficiency LLE', 'Ligand Efficiency SEI', 'Potential Duplicate', 'Assay ChEMBL ID', 'Assay Description', 'Assay Type', 'BAO Format ID', 'BAO Label', 'Assay Organism', 'Assay Tissue ChEMBL ID', 'Assay Tissue Name', 'Assay Cell Type', 'Assay Subcellular Fraction', 'Assay Parameters', 'Assay Variant Accession', 'Assay Variant Mutation', 'Target ChEMBL ID', 'Target Name', 'Target Organism', 'Target Type', 'Document ChEMBL ID', 'Source ID', 'Source Description', 'Document Journal', 'Document Year', 'Cell ChEMBL ID', 'Properties', 'Action Type', 'Standard Text Value', 'Value']


,Molecule ChEMBL ID,Molecule Name,Molecule Max Phase,Molecular Weight,#RO5 Violations,AlogP,Compound Key,Smiles,Standard Type,Standard Relation,...,Document ChEMBL ID,Source ID,Source Description,Document Journal,Document Year,Cell ChEMBL ID,Properties,Action Type,Standard Text Value,Value
14009,CHEMBL244427,NaN,NaN,284.21,0.0,3.16,9,O=C(c1cccc(F)c1F)C(O)c1cccc(F)c1F,Ki,'>',...,CHEMBL1147944,1,Scientific Literature,Bioorg Med Chem,2007.0,NaN,NaN,NaN,NaN,100000.0
10084,CHEMBL4088659,NaN,NaN,312.38,0.0,4.15,3e,CN1c2ccccc2C(n2nnc3ccccc32)c2ccccc21,IC50,'=',...,CHEMBL4043268,1,Scientific Literature,Bioorg Med Chem,2017.0,NaN,NaN,NaN,NaN,12.4
2663,CHEMBL3335027,NaN,NaN,417.55,1.0,5.01,19a3,O=c1oc2cc(OCCN3CCC(Cc4ccccc4)CC3)ccc2c2c1CCCC2,IC50,'=',...,CHEMBL3351711,1,Scientific Literature,Bioorg Med Chem,2014.0,NaN,NaN,NaN,NaN,8.1
19593,CHEMBL325372,METHYLPARABEN,-1.0,152.15,0.0,1.18,methylparaben,COC(=O)c1ccc(O)cc1,AC50,'>',...,CHEMBL5291721,1,Scientific Literature,Nat Commun,2023.0,NaN,NaN,NaN,NaN,30.0
13128,CHEMBL5440621,NaN,NaN,335.30,0.0,2.87,18,CN(C)CCCN1C(=O)CSC1c1ccc(Cl)cc1.Cl,IC50,'=',...,CHEMBL5380867,1,Scientific Literature,J Med Chem,2023.0,NaN,NaN,INHIBITOR,NaN,3.7


In [69]:
# Lets see the data types and missing values in the dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20132 entries, 0 to 20131
Data columns (total 48 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Molecule ChEMBL ID          20132 non-null  str    
 1   Molecule Name               4930 non-null   str    
 2   Molecule Max Phase          4074 non-null   float64
 3   Molecular Weight            20094 non-null  float64
 4   #RO5 Violations             19865 non-null  float64
 5   AlogP                       19865 non-null  float64
 6   Compound Key                20120 non-null  str    
 7   Smiles                      20045 non-null  str    
 8   Standard Type               20132 non-null  str    
 9   Standard Relation           16434 non-null  str    
 10  Standard Value              16373 non-null  float64
 11  Standard Units              16932 non-null  str    
 12  pChEMBL Value               8259 non-null   float64
 13  Data Validity Comment       928 non-null  

**Paso 2: Eliminación de valores nulos tanto de SMILES como de actividad**

Podemos observar que tenemos compuestos con valores nulos para algunas columnas importantes.

Eliminamos estas filas debido a que no poseen su representación moleculár básica y tampoco el valor de actividad biológica de interés en este trabajo.

In [70]:
# Dropping rows with missing values in the 'Smiles' and 'Standard Value' columns
df = df.dropna(subset=['Smiles', 'Standard Value'])

# Saving the size of the dataset after dropping rows with missing values
size_step2 = df.shape

# Printing the size of the dataset after dropping rows with missing values
print(f'🔶 El dataset, después de eliminar filas con valores faltantes tiene {size_step2[0]} filas y {size_step2[1]} columnas')


🔶 El dataset, después de eliminar filas con valores faltantes tiene 16359 filas y 48 columnas


**Paso 3: Limitar los ensayos de bioactividad a IC50**

Tenemos varios tipos de bioactividad (IC50, inhibición, Ki, etc.), sin embargo no podemos comparar ni tratar estos tipos como iguales, ya que cometeríamos un error de base confundiendo los diferentes resultados de experimentación (Afinidad química, bioactividad, cinética, etc.) que se pueden obtener sobre un mismo target.

Debemos tener presente que dichos valores están en la columna 'Standard Type'

In [71]:
# Initial data exploration: Checking the distribution of 'Standard Type' values
df['Standard Type'].value_counts()

Standard Type
IC50          8791
Inhibition    1715
Activity      1339
Ki            1037
AC50           994
              ... 
XC50             1
Kic              1
IC33             1
effect           1
IC               1
Name: count, Length: 74, dtype: int64

In [72]:
# Keeping only the IC50-type activity data
df = df[df['Standard Type'] == 'IC50']
size_step3 = df.shape

# Printing the size of the dataset after filtering by IC50 activity type
print(f'🔶 El dataset, después de filtrar por tipo de actividad IC50 tiene {df.shape[0]} filas y {df.shape[1]} columnas')

🔶 El dataset, después de filtrar por tipo de actividad IC50 tiene 8791 filas y 48 columnas


**Paso 4: Limitar los ensayo a tipo binding (B)**

Realizamos este procedimiento ya que los diferentes tipos de ensayos (Binding, Funcional, ADME y Toxicidad) evaluan objetivos básicos diferentes, como lo son unión, farmacocinética y toxicidad.

Debemos tener presente que dichos valores están en la columna 'Assay Type' (B)

In [73]:
# Initial data exploration: Checking the distribution of 'Assay Type' values
df['Assay Type'].value_counts()

Assay Type
B    8599
A     136
T      37
F      19
Name: count, dtype: int64

In [74]:
# Keeping only the B-type Assay data
df = df[df['Assay Type'] == 'B']
size_step4 = df.shape

# Printing the size of the dataset after filtering by B-type Assay
print(f'🔶 El dataset, después de filtrar por tipo de actividad B tiene {df.shape[0]} filas y {df.shape[1]} columnas')

🔶 El dataset, después de filtrar por tipo de actividad B tiene 8599 filas y 48 columnas


**Paso 5: Limitar los ensayo a aquellos con unidades de nano o micro molar**

Realizamos este procedimiento para obtener valores estandar que se pueden convertir entre si.

Debemos tener presente que dichos valores están en la columna 'Standard Units'

In [75]:
# Initial data exploration: Checking the distribution of 'Standard Units' values
df['Standard Units'].value_counts()

Standard Units
nM             8508
ug.mL-1          83
10'5pM            3
10^-4microM       2
10'6pM            2
10'3pM            1
Name: count, dtype: int64

In [76]:
# keeping only the data with 'nM' as the standard unit (keep in mind that there are not data with 'uM' as the standard unit)
df = df[df['Standard Units'] == 'nM']
size_step5 = df.shape

# Printing the size of the dataset after filtering by 'nM' standard unit
print(f'🔶 El dataset, después de filtrar por unidad nM tiene {df.shape[0]} filas y {df.shape[1]} columnas')

🔶 El dataset, después de filtrar por unidad nM tiene 8508 filas y 48 columnas


**Paso 6: Eliminar duplicados por ChEMBL ID, manteniendo los de mayor actividad biológica**

Realizamos este procedimiento para evitar tener moléculas que tienen el mismo ChEMBL ID -iguales- (Recordar que previamente eliminamos las filas con SMILES nulos, mas no eliminamos aquellas con duplicados) y quedarnos con la de mayor interés (menor IC50).

Debemos tener presente que dichos valores están en la columna 'Molecule ChEMBL ID'

In [77]:
# Initial data exploration: Checking the distribution of 'Standard Units' values
i_size = df.shape[0]
n_unique_molecules = df['Molecule ChEMBL ID'].nunique()

print(f'🔷 El dataset tiene {i_size} filas y {n_unique_molecules} moléculas con ChEMBL ID únicos')

🔷 El dataset tiene 8508 filas y 6939 moléculas con ChEMBL ID únicos


In [78]:
# Visualizing the distribution of 'Molecule ChEMBL ID' values
df['Molecule ChEMBL ID'].value_counts().head(10)

Molecule ChEMBL ID
CHEMBL95        182
CHEMBL502       164
CHEMBL659        86
CHEMBL636        56
CHEMBL94         52
CHEMBL395280     23
CHEMBL292314     19
CHEMBL140476     13
CHEMBL433041     12
CHEMBL345124     12
Name: count, dtype: int64

In [79]:
# We will use the compound 'CHEMBL95' in order to verify our next steps
print(f'🔷 El ensayo con mayor actividad biológica (menor IC50) para la molécula CHEMBL95 es de {df[df["Molecule ChEMBL ID"]=="CHEMBL95"]["Standard Value"].min()} nM')
df[df['Molecule ChEMBL ID']=='CHEMBL95'][['Standard Value']].describe()

🔷 El ensayo con mayor actividad biológica (menor IC50) para la molécula CHEMBL95 es de 3.16 nM


,Standard Value
count,182.000000
mean,259.356538
std,183.942878
min,3.160000
25%,130.250000
50%,226.500000
75%,368.000000
max,1180.000000


In [80]:
# Sorting the dataset by 'Standard Value' 
df = df.sort_values(by='Standard Value', ascending=True)

# Removing duplicates while keeping the first occurrence (better activity) 
df = df.drop_duplicates(subset=['Molecule ChEMBL ID'], keep='first')

size_step6 = df.shape
# Printing the size of the dataset after removing duplicates by ChEMBL ID
print(f'🔶 El dataset, después de eliminar duplicados por ChEMBL ID tiene {df.shape[0]} filas y {df.shape[1]} columnas')

# Verifying the compound 'CHEMBL95' after removing duplicates
print(f'🔷 La actividad biológica para la molécula CHEMBL95 luego de la curación es de {df[df["Molecule ChEMBL ID"]=="CHEMBL95"]["Standard Value"].iloc[0]} nM')

🔶 El dataset, después de eliminar duplicados por ChEMBL ID tiene 6939 filas y 48 columnas
🔷 La actividad biológica para la molécula CHEMBL95 luego de la curación es de 3.16 nM


**Paso 7: Calcular el pIC50**

Realizamos este procedimiento con el fin de simplificar la comparación de la potencia de diferentes compuestos químicos sobre una escala lineal y fácil de interpretar.

Debemos tener presente que dichos valores están en la columna 'Standard Value'


In [81]:
# Initial data exploration: Checking the distribution of 'Standard Value' values
df['Standard Value'].describe()

count    6.939000e+03
mean     9.351286e+04
std      2.111901e+06
min      0.000000e+00
25%      1.442500e+02
50%      2.137960e+03
75%      1.390000e+04
max      1.636817e+08
Name: Standard Value, dtype: float64

**Nota: Tener presente los outliers que se encuentran en esta categoría para su futura curación**

In [82]:
# Creating a function to convert IC50 values from nM to pIC50
def convert_to_pIC50(ic50_value):
    '''
    This function converts IC50 values (parameter) in nM to pIC50 values (return)
    '''
    if ic50_value <= 0:
        return None  # Return None for non-positive IC50 values
    else:
        pIC50 = 9 - math.log10(ic50_value) # Remember to import math library!!
    
    return pIC50

In [83]:
# Creating a new column 'pIC50' in the dataset by applying the conversion function to 'Standard Value'
df['pIC50'] = df['Standard Value'].apply(convert_to_pIC50)

# Dropping rows with None values in the 'pIC50' column
df = df.dropna(subset=['pIC50'])

size_step7 = df.shape
# Printing the size of the dataset after converting IC50 to pIC50 and dropping None values
print(f'🔶 El dataset, después de convertir IC50 a pIC50 y eliminar valores None tiene {df.shape[0]} filas y {df.shape[1]} columnas')

df.sample(5)

🔶 El dataset, después de convertir IC50 a pIC50 y eliminar valores None tiene 6938 filas y 49 columnas


,Molecule ChEMBL ID,Molecule Name,Molecule Max Phase,Molecular Weight,#RO5 Violations,AlogP,Compound Key,Smiles,Standard Type,Standard Relation,...,Source ID,Source Description,Document Journal,Document Year,Cell ChEMBL ID,Properties,Action Type,Standard Text Value,Value,pIC50
11953,CHEMBL304820,NaN,NaN,195.20,0.0,1.61,2f,O/N=C/c1nc(-c2cccs2)no1,IC50,'>',...,1,Scientific Literature,J Med Chem,1986.0,NaN,NaN,NaN,NaN,0.36,3.443697
9526,CHEMBL371787,NaN,NaN,352.43,0.0,3.92,6,CC(NC(=O)Oc1cccc2c1OCC1CCN(C)C21)c1ccccc1,IC50,'=',...,1,Scientific Literature,J Med Chem,2004.0,NaN,NaN,NaN,NaN,1870.00,5.728158
16482,CHEMBL218939,NaN,NaN,363.46,0.0,3.94,11,CCOC(=O)C1=C(C)Nc2nc3c(c(N)c2C1c1ccccc1)CCCC3,IC50,'=',...,1,Scientific Literature,Bioorg Med Chem,2008.0,NaN,NaN,NaN,NaN,0.12,6.920819
4863,CHEMBL3355594,NaN,NaN,258.32,0.0,1.21,5,C/C=C1/[C@@H]2Cc3[nH]c(=O)ccc3[C@@]1(N)C[C@]1(...,IC50,'=',...,1,Scientific Literature,J Nat Prod,2014.0,NaN,NaN,NaN,NaN,12.11,4.916856
13015,CHEMBL4782832,NaN,NaN,526.72,2.0,6.00,TM-19,CCN(CC)CCCCOc1ccc(/C=C/C(=O)c2c(O)cc(OCCCCN(CC...,IC50,'=',...,1,Scientific Literature,Eur J Med Chem,2020.0,NaN,NaN,INHIBITOR,NaN,4.70,5.327902


**Paso 8: Guardar el resultado en formato .tsv**

Realizamos este procedimiento con el fin de guardar nuestro dataset para futuros trabajos.

In [84]:
# Saving the curated dataset to a new .tSV file
df.to_csv('ache_chembl220_curated.tsv', sep='\t', index=False)

# **Resumen**

Se realizó una curación inicial de 8 pasos para el dataset obtenido de ChEMBL para el target Acetylcholinesterase (CHEMBL220):

1. *Cargar el dataset (archivo .tsv) y análisis exploratio inicial de los datos*
2. *Eliminación de valores nulos tanto de SMILES como de actividad*
3. *Limitar los ensayos de bioactividad a IC50*
4. *Limitar los ensayo a tipo binding (B)*
5. *Limitar los ensayo a aquellos con unidades de nano o micro molar*
6. *Eliminar duplicados por ChEMBL ID, manteniendo los de mayor actividad biológica*
7. *Calcular el pIC50*
8. *Guardar el resultado en formato .tsv*

Consideraciones:

- No se evidenciaron compuestos con bioactividad en unidades uM
- Tener en cuenta los valores atipícos de actividad biológica (incluyendo el compuesto con 0 nM)
- Los outliers se podrían evidenciar de una mejor manera usando gráficós de dispersión


In [85]:
# Final report of the dataset after all the curation steps
print(f'📋 Tamaño inicial del dataset: {size_step1}')
print(f'📋 Tamaño después de eliminar filas con valores faltantes: {size_step2}')
print(f'📋 Tamaño después de filtrar por tipo de actividad IC50: {size_step3}')
print(f'📋 Tamaño del dataset después de filtrar por tipo de actividad B: {size_step4}')
print(f'📋 Tamaño después de filtrar por unidad nM: {size_step5}')
print(f'📋 Tamaño después de eliminar duplicados por ChEMBL ID: {size_step6}')
print(f'📋 Tamaño final del dataset después de convertir IC50 a pIC50 y eliminar valores None: {size_step7}')


📋 Tamaño inicial del dataset: (20132, 48)
📋 Tamaño después de eliminar filas con valores faltantes: (16359, 48)
📋 Tamaño después de filtrar por tipo de actividad IC50: (8791, 48)
📋 Tamaño del dataset después de filtrar por tipo de actividad B: (8599, 48)
📋 Tamaño después de filtrar por unidad nM: (8508, 48)
📋 Tamaño después de eliminar duplicados por ChEMBL ID: (6939, 48)
📋 Tamaño final del dataset después de convertir IC50 a pIC50 y eliminar valores None: (6938, 49)
